# Crawling Data Audio

Notebook ini digunakan untuk mengumpulkan data audio dari berbagai sumber.

**Sumber yang tersedia:**
1. FreeSound API
2. YouTube (via yt-dlp)
3. Dataset publik (ESC-50, UrbanSound8K)

> ⚠️ Pastikan dependencies sudah terinstall: `pip install -r ../requirements.txt`

In [ ]:
import os
import requests
import json
import shutil
import time
from pathlib import Path
from tqdm import tqdm

# ============================================================
# KONFIGURASI - Edit bagian ini sesuai kebutuhan
# ============================================================

# Daftar kelas audio yang ingin dikumpulkan
# Contoh: ['rain', 'wind', 'thunder', 'dog', 'cat']
AUDIO_CLASSES = [
    'class_1',  # Ganti dengan nama kelas sebenarnya
    'class_2',
    'class_3',
]

# Jumlah sampel per kelas
SAMPLES_PER_CLASS = 50

# Direktori output
OUTPUT_DIR = Path('../data/raw')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Konfigurasi:')
print(f'  Kelas: {AUDIO_CLASSES}')
print(f'  Sampel per kelas: {SAMPLES_PER_CLASS}')
print(f'  Output: {OUTPUT_DIR.resolve()}')

## Metode 1: FreeSound API

FreeSound adalah database audio open-source dengan ribuan file gratis.

**Langkah-langkah:**
1. Daftar di [freesound.org](https://freesound.org/apiv2/apply)
2. Buat aplikasi dan dapatkan API Key
3. Masukkan API Key di bawah ini

In [ ]:
# ============================================================
# FREESOUND API
# ============================================================

FREESOUND_API_KEY = 'MASUKKAN_API_KEY_DISINI'  # <-- Ganti ini
FREESOUND_BASE_URL = 'https://freesound.org/apiv2'

def search_freesound(query, api_key, page_size=15, min_duration=1.0, max_duration=30.0):
    """Cari audio di FreeSound berdasarkan query."""
    url = f'{FREESOUND_BASE_URL}/search/text/'
    params = {
        'query': query,
        'token': api_key,
        'page_size': page_size,
        'fields': 'id,name,duration,previews,license',
        'filter': f'duration:[{min_duration} TO {max_duration}]',
        'format': 'json'
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json().get('results', [])
    else:
        print(f'Error {response.status_code}: {response.text}')
        return []

def download_freesound_audio(sound, output_dir, api_key):
    """Download satu file audio dari FreeSound."""
    preview_url = sound.get('previews', {}).get('preview-hq-mp3')
    if not preview_url:
        return False
    
    # Tambahkan token ke URL
    download_url = f"{preview_url}?token={api_key}"
    filename = f"{sound['id']}_{sound['name'][:30].replace('/', '_')}.mp3"
    filepath = output_dir / filename
    
    if filepath.exists():
        return True  # Skip jika sudah ada
    
    response = requests.get(download_url, stream=True)
    if response.status_code == 200:
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    return False

def crawl_freesound(audio_classes, api_key, output_dir, samples_per_class=50):
    """Crawl audio dari FreeSound untuk semua kelas."""
    if api_key == 'MASUKKAN_API_KEY_DISINI':
        print('⚠️ Harap masukkan API Key FreeSound terlebih dahulu!')
        return
    
    for class_name in audio_classes:
        class_dir = output_dir / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        
        print(f'\n[FreeSound] Mencari: {class_name}')
        sounds = search_freesound(class_name, api_key, page_size=samples_per_class)
        
        success = 0
        for sound in tqdm(sounds, desc=f'Download {class_name}'):
            if download_freesound_audio(sound, class_dir, api_key):
                success += 1
            time.sleep(0.5)  # Rate limiting
        
        print(f'  Berhasil download: {success}/{len(sounds)} file')

# Jalankan crawling FreeSound
crawl_freesound(AUDIO_CLASSES, FREESOUND_API_KEY, OUTPUT_DIR, SAMPLES_PER_CLASS)

## Metode 2: YouTube (via yt-dlp)

Download audio dari URL YouTube atau playlist.

In [ ]:
# ============================================================
# YOUTUBE CRAWLING
# ============================================================

import subprocess

# Daftar URL YouTube per kelas
# Format: {'nama_kelas': ['url1', 'url2', ...]}
YOUTUBE_URLS = {
    'class_1': [
        # 'https://www.youtube.com/watch?v=XXXX',
        # 'https://www.youtube.com/playlist?list=XXXX',
    ],
    'class_2': [
        # 'https://www.youtube.com/watch?v=XXXX',
    ],
}

def download_youtube_audio(urls, class_name, output_dir):
    """Download audio dari YouTube menggunakan yt-dlp."""
    if not urls:
        print(f'Tidak ada URL untuk kelas: {class_name}')
        return
    
    class_dir = output_dir / class_name
    class_dir.mkdir(parents=True, exist_ok=True)
    
    for url in urls:
        print(f'\nDownload dari: {url}')
        cmd = [
            'yt-dlp',
            '-x',                          # Extract audio only
            '--audio-format', 'wav',        # Format WAV
            '--audio-quality', '0',         # Best quality
            '-o', str(class_dir / '%(title)s.%(ext)s'),
            '--no-playlist',               # Hapus ini jika ingin download playlist
            url
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f'  ✅ Berhasil')
        else:
            print(f'  ❌ Gagal: {result.stderr[:200]}')

for class_name, urls in YOUTUBE_URLS.items():
    download_youtube_audio(urls, class_name, OUTPUT_DIR)

## Metode 3: Dataset Publik (ESC-50)

Download dataset ESC-50 yang berisi 2000 file audio untuk 50 kategori lingkungan.

In [ ]:
# ============================================================
# DOWNLOAD ESC-50 DATASET
# ============================================================

import zipfile
import pandas as pd

# Kategori ESC-50 yang ingin digunakan
# Lihat daftar lengkap di: https://github.com/karolpiczak/ESC-50
ESC50_CATEGORIES = [
    # Contoh: 'rain', 'sea_waves', 'crackling_fire', 'dog', 'cat'
    # Ganti dengan kategori yang diinginkan
]

ESC50_URL = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
ESC50_ZIP = Path('../data/ESC-50.zip')
ESC50_DIR = Path('../data/ESC-50-master')

def download_esc50():
    """Download dataset ESC-50."""
    if not ESC50_ZIP.exists():
        print('Downloading ESC-50...')
        response = requests.get(ESC50_URL, stream=True)
        total = int(response.headers.get('content-length', 0))
        with open(ESC50_ZIP, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
        print('✅ Download selesai!')
    else:
        print('ESC-50 sudah didownload.')

def extract_esc50_categories(categories, output_dir):
    """Ekstrak file audio ESC-50 berdasarkan kategori."""
    if not ESC50_DIR.exists():
        print('Mengekstrak zip...')
        with zipfile.ZipFile(ESC50_ZIP) as zf:
            zf.extractall(Path('../data'))
    
    meta_path = ESC50_DIR / 'meta' / 'esc50.csv'
    meta = pd.read_csv(meta_path)
    
    for category in categories:
        subset = meta[meta['category'] == category]
        if subset.empty:
            print(f'⚠️ Kategori tidak ditemukan: {category}')
            print(f'   Tersedia: {meta["category"].unique().tolist()}')
            continue
        
        cat_dir = output_dir / category
        cat_dir.mkdir(parents=True, exist_ok=True)
        
        for _, row in subset.iterrows():
            src = ESC50_DIR / 'audio' / row['filename']
            dst = cat_dir / row['filename']
            if src.exists() and not dst.exists():
                shutil.copy(src, dst)
        
        print(f'✅ {category}: {len(subset)} file disalin ke {cat_dir}')

if ESC50_CATEGORIES:
    download_esc50()
    extract_esc50_categories(ESC50_CATEGORIES, OUTPUT_DIR)
else:
    print('Tidak ada kategori ESC-50 yang dipilih. Edit variabel ESC50_CATEGORIES di atas.')

## Ringkasan Hasil Crawling

In [ ]:
# ============================================================
# RINGKASAN HASIL
# ============================================================

print('=' * 50)
print('RINGKASAN HASIL CRAWLING')
print('=' * 50)

total_files = 0
summary = []

if OUTPUT_DIR.exists():
    for class_dir in sorted(OUTPUT_DIR.iterdir()):
        if class_dir.is_dir():
            audio_extensions = ('.wav', '.mp3', '.flac', '.ogg', '.m4a')
            files = [f for f in class_dir.iterdir() if f.suffix.lower() in audio_extensions]
            count = len(files)
            total_files += count
            summary.append({'Kelas': class_dir.name, 'Jumlah File': count})
            print(f'  {class_dir.name}: {count} file')

print(f'\nTotal: {total_files} file audio')

if summary:
    df_summary = pd.DataFrame(summary) if 'pd' in dir() else None
    if df_summary is not None:
        print('\n', df_summary.to_string(index=False))